# 📥 Etapa 1: Ingesta, Curaduría de Datos y Construcción de SQLite
**Observatorio Ciudadano de Contratación Pública (SECOP II)**

En esta etapa descargamos los contratos públicos del portal de datos abiertos (`datos.gov.co`), aplicamos anonimización estricta sobre documentos de personas naturales y construimos la base de datos relacional local (`secop_contratos.sqlite`) con índices de consulta rápida.

In [27]:
# Recarga automática de módulos en src/observatorio/ al editar archivos .py
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Asegurar que `src/` esté en sys.path incluso si se abre desde `notebooks/`
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from observatorio.config import (
    CLEAN_CSV_PATH,
    RANDOM_SEED,
    RAW_CSV_PATH,
    RAW_METADATA_PATH,
    SECOP_DEPARTAMENTO,
    SECOP_FECHA_FIN,
    SECOP_FECHA_INICIO,
    SQLITE_DB_PATH,
)
from observatorio.data.ingestion import COLUMNAS_SELECCIONADAS, descargar_subconjunto_secop
from observatorio.data.cleaning import (
    generar_resumen_validacion,
    limpiar_contratos_secop,
    obtener_diccionario_columnas,
)
from observatorio.data.database import ejecutar_consulta_segura, guardar_en_sqlite

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

print("Entorno configurado correctamente.")
print(f"Departamento objetivo : {SECOP_DEPARTAMENTO}")
print(f"Rango de fechas       : {SECOP_FECHA_INICIO}  -->  {SECOP_FECHA_FIN}")
print(f"Ruta CSV Original     : {RAW_CSV_PATH}")
print(f"Ruta CSV Limpio       : {CLEAN_CSV_PATH}")
print(f"Ruta Base SQLite      : {SQLITE_DB_PATH}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Entorno configurado correctamente.
Departamento objetivo : Valle del Cauca
Rango de fechas       : 2023-01-01T00:00:00  -->  2026-10-31T23:59:59
Ruta CSV Original     : /Volumes/Content/Proyects/IA (Artificial Inteligen)/observatorio_secop_ii_desarrollo/data/raw/secop_contratos_original.csv
Ruta CSV Limpio       : /Volumes/Content/Proyects/IA (Artificial Inteligen)/observatorio_secop_ii_desarrollo/data/processed/secop_contratos_limpio.csv
Ruta Base SQLite      : /Volumes/Content/Proyects/IA (Artificial Inteligen)/observatorio_secop_ii_desarrollo/data/processed/secop_contratos.sqlite


---
# 📦 PARTE 1: De los Datos Abiertos a una Base Confiable (SECOP II)

## 1.1 Conexión Directa con Datos Abiertos Colombia
En lugar de trabajar con archivos estáticos descargados a mano, consultamos directamente la API oficial de **Datos Abiertos (`datos.gov.co`)**. Acotamos la descarga a un departamento y una ventana de tiempo concreta, registrando la fecha exacta de corte (ya que SECOP II cambia todos los días) y guardando una copia intacta del archivo original (`secop_contratos_original.csv`) como respaldo para auditoría.

In [28]:
# Cambia FORZAR_DESCARGA = True cuando quieras volver a consultar la API de datos.gov.co
FORZAR_DESCARGA = False
MAX_REGISTROS_DESCARGA = (
    50000  # Ajustable entre 20.000 y 100.000 según alcance (Sección 3)
)

if RAW_CSV_PATH.exists() and RAW_METADATA_PATH.exists() and not FORZAR_DESCARGA:
    print(f"📂 Cargando CSV original existente desde: {RAW_CSV_PATH}")
    df_raw = pd.read_csv(RAW_CSV_PATH, low_memory=False)
    metadatos_descarga = json.loads(RAW_METADATA_PATH.read_text(encoding="utf-8"))
else:
    print("🌐 Descargando subconjunto desde la API Socrata (datos.gov.co)...")
    df_raw, metadatos_descarga = descargar_subconjunto_secop(
        departamento=SECOP_DEPARTAMENTO,
        fecha_inicio=SECOP_FECHA_INICIO,
        fecha_fin=SECOP_FECHA_FIN,
        batch_size=10000,
        max_records=MAX_REGISTROS_DESCARGA,
        guardar_csv=True,
        ruta_csv=RAW_CSV_PATH,
        ruta_metadatos=RAW_METADATA_PATH,
    )
    print(f"✅ CSV original guardado en: {RAW_CSV_PATH}")

print("\n📋 Metadatos de trazabilidad de la descarga:")
for k, v in metadatos_descarga.items():
    if k != "columnas_descargadas":
        print(f"  - {k}: {v}")

print(
    f"\n🔎 Vista previa del CSV Original ({df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas):"
)
display(df_raw.head(5))

📂 Cargando CSV original existente desde: /Volumes/Content/Proyects/IA (Artificial Inteligen)/observatorio_secop_ii_desarrollo/data/raw/secop_contratos_original.csv

📋 Metadatos de trazabilidad de la descarga:
  - fecha_consulta_utc: 2026-10-30T06:00:29.008538+00:00
  - departamento: Valle del Cauca
  - fecha_inicio: 2023-01-01T00:00:00
  - fecha_fin: 2026-10-31T23:59:59
  - total_registros_descargados: 50000
  - dataset_id: jbjy-vk9h
  - ruta_csv_original: /Volumes/Content/Proyects/IA (Artificial Inteligen)/observatorio_secop_ii_desarrollo/data/raw/secop_contratos_original.csv

🔎 Vista previa del CSV Original (50,000 filas x 25 columnas):


,id_contrato,nombre_entidad,departamento,ciudad,sector,rama,orden,estado_contrato,tipo_de_contrato,modalidad_de_contratacion,fecha_de_firma,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato,valor_del_contrato,dias_adicionados,es_pyme,es_grupo,habilita_pago_adelantado,origen_de_los_recursos,codigo_de_categoria_principal,tipodocproveedor,proveedor_adjudicado,descripcion_del_proceso,objeto_del_contrato,urlproceso
0,CO1.PCCNTR.10000004,DEPARTAMENTO DEL VALLE DEL CAUCA-SECRETARIA DE GESTION DEL RIESGO DE DESASTRES,Valle del Cauca,Cali,"Vivienda, Ciudad y Territorio",Ejecutivo,Territorial,En ejecución,Prestación de servicios,Contratación directa,2026-09-28T17:58:34.043,2026-09-29T05:00:00.000,2026-12-31T23:00:00.000,10000000.0,0,Si,No,No,Distribuido,UNSPECIFIED,Cédula de Ciudadanía,Yanneth Henao Ledesma,PRESTAR LOS SERVICIOS DE APOYO A LA GESTIÓN EN LA ACTIVIDAD ORIENTAR EN LA F...,PRESTAR LOS SERVICIOS DE APOYO A LA GESTIÓN EN LA ACTIVIDAD ORIENTAR EN LA F...,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/I...
1,CO1.PCCNTR.10000013,DEPARTAMENTO DEL VALLE DEL CAUCA - OFICINA PARA LA TRANSPARENCIA DE LA GESTI...,Valle del Cauca,Cali,Servicio Público,Ejecutivo,Territorial,En ejecución,Prestación de servicios,Contratación directa,2026-09-28T15:08:16.759,2026-09-29T08:00:00.000,2026-10-31T23:59:00.000,15400000.0,0,No,No,No,Distribuido,UNSPECIFIED,Cédula de Ciudadanía,ALIRIA DEL CARMEN GONALEZ VALENCIA,PRESTACIÓN DE SERVICIOS PROFESIONALES COMO ADMINISTRADOR DE EMPRESAS; MAGIST...,PRESTACIÓN DE SERVICIOS PROFESIONALES COMO ADMINISTRADOR DE EMPRESAS; MAGIST...,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/I...
2,CO1.PCCNTR.10000018,ALCALDIA MUNICIPIO DE JAMUNDI,Valle del Cauca,Jamundí,No aplica/No pertenece,Ejecutivo,Territorial,Modificado,Prestación de servicios,Contratación directa,2026-09-28T13:35:37.860,2026-09-28T14:00:00.000,2026-12-31T23:00:00.000,12000000.0,0,No,No,No,Distribuido,UNSPECIFIED,Cédula de Ciudadanía,Andres Jose urbano Rodriguez,PRESTACION DE SERVICIOS DE APOYO A LA\nGESTION EN LA SECRETARÍA DE HACIENDA;...,PRESTACION DE SERVICIOS DE APOYO A LA\nGESTION EN LA SECRETARÍA DE HACIENDA;...,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/I...
3,CO1.PCCNTR.10000020,DEPARTAMENTO DEL VALLE DEL CAUCA-SECRETARIA DE GESTION DEL RIESGO DE DESASTRES,Valle del Cauca,Cali,"Vivienda, Ciudad y Territorio",Ejecutivo,Territorial,En ejecución,Prestación de servicios,Contratación directa,2026-09-28T16:29:44.152,2026-09-29T05:00:00.000,2026-12-31T23:00:00.000,14000000.0,0,No,No,No,Distribuido,UNSPECIFIED,Cédula de Ciudadanía,Luz Mary Ramirez Marin,PRESTAR LOS SERVICIOS DE APOYO A LA GESTIÓN COMO TECNOLOGA EN PREVENCION Y A...,PRESTAR LOS SERVICIOS DE APOYO A LA GESTIÓN COMO TECNOLOGA EN PREVENCION Y A...,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/I...
4,CO1.PCCNTR.10000032,DEPARTAMENTO DEL VALLE DEL CAUCA-SECRETARIA DE GESTION DEL RIESGO DE DESASTRES,Valle del Cauca,Cali,"Vivienda, Ciudad y Territorio",Ejecutivo,Territorial,En ejecución,Prestación de servicios,Contratación directa,2026-09-28T16:16:44.601,2026-09-29T05:00:00.000,2026-12-31T23:00:00.000,24000000.0,0,No,No,No,Distribuido,UNSPECIFIED,Cédula de Ciudadanía,STEPHANY SALAS BASANTE,PRESTAR LOS SERVICIOS PROFESIONALES COMO ABOGADA ESPECIALISTA Y MAGISTER EN ...,PRESTAR LOS SERVICIOS PROFESIONALES COMO ABOGADA ESPECIALISTA Y MAGISTER EN ...,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/I...


## 1.2 Curaduría de Datos y Protección de Privacidad
Los registros administrativos crudos suelen incluir ruido, formatos inconsistentes y datos sensibles. En este paso preparamos una versión limpia (`secop_contratos_limpio.csv`) aplicando tres criterios:
- **Privacidad por diseño:** Excluimos documentos de identidad, direcciones, cuentas bancarias y anonimizamos los nombres de contratistas cuando corresponden a personas naturales, manteniendo visibles únicamente las razones sociales de empresas (`NIT`).
- **Calidad y consistencia:** Eliminamos registros duplicados por `id_contrato`, estandarizamos fechas y valores monetarios, y extraemos el enlace directo al proceso original.
- **Contraste transparente:** Comparamos el archivo original frente al archivo limpio para verificar qué cambió exactamente durante el proceso.

In [29]:
# Ejecutar limpieza y guardar el archivo CSV limpio en data/processed/secop_contratos_limpio.csv
df_clean = limpiar_contratos_secop(
    df_raw,
    guardar_csv=True,
    ruta_csv_limpio=CLEAN_CSV_PATH,
)

print(f"✅ CSV limpio guardado en: {CLEAN_CSV_PATH}")

# 1. Tabla comparativa de validación (CSV Original vs CSV Limpio)
print("\n📊 Tabla de Validación Comparativa (Original vs. Limpio):")
df_resumen_validacion = generar_resumen_validacion(df_raw, df_clean)
display(df_resumen_validacion)

# 2. Verificación visual de la transformación en columnas clave
cols_comparacion = [
    "id_contrato",
    "fecha_de_firma",
    "valor_del_contrato",
    "dias_adicionados",
    "tipodocproveedor",
    "proveedor_adjudicado",
    "urlproceso",
]
cols_presentes = [c for c in cols_comparacion if c in df_clean.columns]
print("\n🔎 Muestra del CSV Limpio en columnas auditadas:")
display(df_clean[cols_presentes].head(5))

# 3. Diccionario de las 25 columnas seleccionadas
print("\n📖 Diccionario de Columnas del Proyecto:")
display(obtener_diccionario_columnas())

✅ CSV limpio guardado en: /Volumes/Content/Proyects/IA (Artificial Inteligen)/observatorio_secop_ii_desarrollo/data/processed/secop_contratos_limpio.csv

📊 Tabla de Validación Comparativa (Original vs. Limpio):


,Métrica de Validación,CSV Original (Raw),CSV Limpio (Processed)
0,Total de filas (registros),50000,50000
1,Total de columnas,25,25
2,Duplicados por id_contrato,0,0
3,Proveedores Persona Natural anonimizados,Sin anonimizar (0),46930
4,Nulos en valor_del_contrato,0,0
5,Nulos en dias_adicionados,0,0



🔎 Muestra del CSV Limpio en columnas auditadas:


,id_contrato,fecha_de_firma,valor_del_contrato,dias_adicionados,tipodocproveedor,proveedor_adjudicado,urlproceso
0,CO1.PCCNTR.10000004,2026-09-28,10000000.0,0,Cédula de Ciudadanía,PERSONA NATURAL (ANONIMIZADO),{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/I...
1,CO1.PCCNTR.10000013,2026-09-28,15400000.0,0,Cédula de Ciudadanía,PERSONA NATURAL (ANONIMIZADO),{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/I...
2,CO1.PCCNTR.10000018,2026-09-28,12000000.0,0,Cédula de Ciudadanía,PERSONA NATURAL (ANONIMIZADO),{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/I...
3,CO1.PCCNTR.10000020,2026-09-28,14000000.0,0,Cédula de Ciudadanía,PERSONA NATURAL (ANONIMIZADO),{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/I...
4,CO1.PCCNTR.10000032,2026-09-28,24000000.0,0,Cédula de Ciudadanía,PERSONA NATURAL (ANONIMIZADO),{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/I...



📖 Diccionario de Columnas del Proyecto:


,columna,tipo,uso
0,id_contrato,Texto (PK),Identificador único del contrato electrónico en SECOP II.
1,nombre_entidad,Categórica,Nombre oficial de la entidad estatal contratante.
2,departamento,Categórica,Departamento de ejecución o registro de la entidad.
3,ciudad,Categórica,Municipio o ciudad asociada a la entidad contratante.
4,sector,Categórica,Sector administrativo al que pertenece la entidad.
5,rama,Categórica,"Rama del poder público (Ejecutiva, Legislativa, Judicial, etc.)."
6,orden,Categórica,"Nivel administrativo (Nacional, Territorial)."
7,estado_contrato,Categórica,"Estado actual del contrato (solo para consultas SQL, excluido de ML por leak..."
8,tipo_de_contrato,Categórica,"Tipología contractual (Prestación de servicios, Obra, Suministro, etc.)."
9,modalidad_de_contratacion,Categórica,"Modalidad legal de selección (Contratación directa, Licitación, Mínima cuant..."


## 1.3 Construcción de la Base de Datos de Consulta Rápida (`SQLite`)
Con el dataset curado y validado, lo estructuramos dentro de una base de datos relacional local (`secop_contratos.sqlite`). Esta base sirve como fuente única tanto para responder preguntas sobre cifras en el asistente conversacional como para alimentar el entrenamiento del modelo predictivo.

In [30]:
# Volcar el DataFrame limpio en SQLite y crear índices
total_en_sqlite = guardar_en_sqlite(df_clean, db_path=SQLITE_DB_PATH)

assert total_en_sqlite == len(
    df_clean
), f"Discrepancia de conteo: SQLite ({total_en_sqlite}) != CSV Limpio ({len(df_clean)})"
print(f"✅ Base de datos SQLite creada e indexada en: {SQLITE_DB_PATH}")
print(
    f"✔️ Conteo verificado: {total_en_sqlite:,} contratos en SQLite == {len(df_clean):,} en CSV limpio."
)

# Prueba rápida del ejecutor SQL en modo solo lectura (modo seguro)
consulta_prueba = """
SELECT modalidad_de_contratacion,
       COUNT(*) AS total_contratos,
       ROUND(AVG(valor_del_contrato), 0) AS valor_promedio_cop
FROM contratos
GROUP BY modalidad_de_contratacion
ORDER BY total_contratos DESC
"""
df_sql_test, sql_ejecutado = ejecutar_consulta_segura(
    consulta_prueba, db_path=SQLITE_DB_PATH
)
print(f"\n🔒 SQL ejecutado en modo solo lectura:\n{sql_ejecutado}")
display(df_sql_test)

✅ Base de datos SQLite creada e indexada en: /Volumes/Content/Proyects/IA (Artificial Inteligen)/observatorio_secop_ii_desarrollo/data/processed/secop_contratos.sqlite
✔️ Conteo verificado: 50,000 contratos en SQLite == 50,000 en CSV limpio.

🔒 SQL ejecutado en modo solo lectura:
SELECT modalidad_de_contratacion,
       COUNT(*) AS total_contratos,
       ROUND(AVG(valor_del_contrato), 0) AS valor_promedio_cop
FROM contratos
GROUP BY modalidad_de_contratacion
ORDER BY total_contratos DESC LIMIT 100


,modalidad_de_contratacion,total_contratos,valor_promedio_cop
0,Contratación directa,46449,3.127772e+07
1,Contratación régimen especial,2206,2.023820e+08
2,Mínima cuantía,786,3.485596e+07
3,Contratación Directa (con ofertas),159,1.342916e+09
4,Contratación régimen especial (con ofertas),137,7.665481e+08
5,Selección Abreviada de Menor Cuantía,133,3.886737e+08
6,Selección abreviada subasta inversa,47,1.237557e+09
7,Licitación pública,41,7.309987e+09
8,Concurso de méritos abierto,24,8.685264e+08
9,Licitación pública Obra Publica,13,4.305837e+09
